<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_04_quantisation_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_06 · Notebook 04 — Making It Fit on the Device

**Paired with L6.2 · Post-Training and Reinforcement Learning**

Everything so far has been trained on a laptop and evaluated on a laptop. The
model from notebook 03 is meant to run on a condition-monitoring box bolted to
a machine — a microcontroller-class device with a few hundred kilobytes of
flash and no floating-point unit worth the name.

Post-training quantisation is the cheapest thing you can do about that: take a
trained network and store its weights as 8-bit integers instead of 32-bit
floats. No retraining, four lines of code, roughly a quarter of the size.

## What you will do

1. Measure the baseline honestly: size, latency, accuracy.
2. Quantise it and measure the same three things.
3. Compare against the alternative nobody tries first — just using a smaller
   network.
4. Learn what these numbers do *not* tell you.

## A warning about the timings

You are measuring on a shared laptop, probably with a browser open. Latency
here is noisy and PyTorch's CPU int8 kernels are tuned for far larger matrices
than these. **Quote ratios measured in one session, never absolute numbers**,
and do not be surprised if quantisation makes a network this small *slower*.
That result is real, and section 5 explains it.

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_6_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex06-training-lab/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import Ex_6_core as core

core.set_seed(0)
print("torch", torch.__version__, "| output dir:", core.OUTPUT_DIR)

## 1 · The model to deploy

Notebook 03's fine-tuned classifier, retrained here so this notebook stands on
its own. A wider body than notebook 03 used, because quantisation on a network
of thirty parameters measures nothing but overhead.

In [ ]:
XB, yB = core.machine_b_dataset()
X_few, y_few, X_rest, y_rest = core.few_shot_split(XB, yB, per_class=10, seed=5)

core.set_seed(0)
model = nn.Sequential(nn.Linear(2, 64), nn.Tanh(),
                      nn.Linear(64, 64), nn.Tanh(),
                      nn.Linear(64, 3))
opt = torch.optim.Adam(model.parameters(), lr=0.02)
loss_fn = nn.CrossEntropyLoss()
Xf, yf = torch.tensor(X_few), torch.tensor(y_few)
for _ in range(800):
    opt.zero_grad()
    loss_fn(model(Xf), yf).backward()
    opt.step()

def accuracy(m, X, y):
    m.eval()
    with torch.no_grad():
        return float((m(torch.tensor(X)).argmax(dim=1).numpy() == y).mean())

print("parameters:", core.count_parameters(model))
print(f"held-out accuracy: {accuracy(model, X_rest, y_rest):.3f}")

## 2 · Three numbers, before

Size, latency and accuracy. Any deployment argument that quotes fewer than all
three is hiding something.

In [ ]:
X_bench = torch.tensor(X_rest)

base = {
    "parameters": core.count_parameters(model),
    "bytes": core.model_size_bytes(model),
    "seconds": core.time_forward(model, X_bench),
    "accuracy": accuracy(model, X_rest, y_rest),
}
print(f"parameters : {base['parameters']}")
print(f"size       : {base['bytes']:,} bytes")
print(f"latency    : {base['seconds']*1e6:.1f} us for {len(X_rest)} samples")
print(f"accuracy   : {base['accuracy']:.3f}")
print()
print(f"4 bytes x {base['parameters']} parameters = {4*base['parameters']:,} bytes")
print(f"the file is larger by {base['bytes'] - 4*base['parameters']:,} bytes "
      "-- tensor names, shapes and the pickle framing")

## 3 · Quantise

`quantize_dynamic` replaces every `nn.Linear` with a version that stores its
weight as int8 plus a scale factor, and converts back on the fly during the
forward pass. *Dynamic* means the activations are quantised at runtime from
their observed range, so — unlike static quantisation — it needs no calibration
data. That is why it is the one you reach for first.

### Your turn

In [ ]:
# TODO 1 --- quantise, and measure the same three numbers --------------------------------------------
# Two `...` to replace:
#   line 1  ->  quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
#   line 2  ->  accuracy(qmodel, X_rest, y_rest)
try:
    from torch.ao.quantization import quantize_dynamic
except ImportError:                              # older PyTorch
    from torch.quantization import quantize_dynamic

qmodel = ...                                      # <- quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)

quant = {
    "bytes":    core.model_size_bytes(qmodel),
    "seconds":  core.time_forward(qmodel, X_bench),
    "accuracy": ...,                              # <- accuracy(qmodel, X_rest, y_rest)
}
print("count_parameters on the quantised model says:", core.count_parameters(qmodel),
      "  <- not the real number; its weights are no longer nn.Parameter objects")
# ------------------------------------------------------------------------------

In [ ]:
print(core.error_table(
    [["size (bytes)", f"{base['bytes']:,}", f"{quant['bytes']:,}",
      f"{base['bytes']/quant['bytes']:.2f}x smaller"],
     ["latency (us)", f"{base['seconds']*1e6:.1f}", f"{quant['seconds']*1e6:.1f}",
      f"{base['seconds']/quant['seconds']:.2f}x"],
     ["accuracy", f"{base['accuracy']:.3f}", f"{quant['accuracy']:.3f}",
      f"{quant['accuracy']-base['accuracy']:+.3f}"]],
    ["", "float32", "int8", "change"]))

**What you should see.** Size down by something approaching four times, and
accuracy essentially unchanged — a change of a percentage point or two in
either direction on a 150-sample test set is noise, not evidence.

Latency is the interesting one. It may be *worse*. At this size the per-call
overhead of quantising and dequantising activations swamps the cheaper
arithmetic. Quantisation pays for itself on large matrix multiplications, and
these are 64 by 64.

Write down whichever you got, with the caveat. A result that contradicts the
brochure is still a result.

## 4 · The comparison that keeps you honest

Before claiming quantisation as the win: would a *smaller float model* have
done just as well? It is the obvious alternative and it is almost never in the
table.

### Your turn

In [ ]:
# TODO 2 --- float32 at several widths --------------------------------------------------------------
# Two `...` to replace, inside the loop:
#   line 1  ->  nn.Sequential(nn.Linear(2, w), nn.Tanh(), nn.Linear(w, w), nn.Tanh(), nn.Linear(w, 3))
#   line 2  ->  core.model_size_bytes(m)
WIDTHS = [4, 8, 16, 32, 64]

small = {"width": WIDTHS, "bytes": [], "accuracy": []}
for w in WIDTHS:
    core.set_seed(0)
    m = ...                                       # <- nn.Sequential(nn.Linear(2, w), nn.Tanh(), nn.Linear(w, w), nn.Tanh(), nn.Linear(w, 3))
    opt_w = torch.optim.Adam(m.parameters(), lr=0.02)
    for _ in range(800):
        opt_w.zero_grad()
        loss_fn(m(Xf), yf).backward()
        opt_w.step()
    small["bytes"].append(...)                    # <- core.model_size_bytes(m)
    small["accuracy"].append(accuracy(m, X_rest, y_rest))
# ------------------------------------------------------------------------------

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.plot(small["bytes"], small["accuracy"], "o-", lw=1.9, ms=7,
        color="#1f77b4", label="float32, various widths")
for w, b, a in zip(small["width"], small["bytes"], small["accuracy"]):
    ax.annotate(f"w={w}", (b, a), textcoords="offset points", xytext=(6, -10),
                fontsize=8, color="#1f77b4")
ax.plot([quant["bytes"]], [quant["accuracy"]], "*", ms=18, color="#d94f2b",
        label="int8, width 64")
ax.plot([base["bytes"]], [base["accuracy"]], "s", ms=9, mfc="none", mew=2.0,
        mec="#111111", label="float32, width 64")
ax.set_xscale("log")
ax.set_xlabel("serialised size [bytes]"); ax.set_ylabel("held-out accuracy")
ax.set_title("Accuracy against size — quantisation is one point on a curve")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, which="both")
plt.show()

**What you should see.** A curve that rises steeply and then flattens, with the
quantised point sitting to the left of the width-64 float point at roughly the
same height.

Now read it as an engineer. If a narrow float model reaches the same accuracy
in fewer bytes than your quantised wide one, quantisation was the wrong lever —
you had capacity you were not using, and the honest fix is a smaller model.
Quantisation earns its place when you are **on the flat part of the curve and
still too big**, which for a real network usually means after you have already
chosen the smallest architecture that works.

This is why the comparison belongs in the table. "We quantised and got 4x
smaller" is only impressive if the model needed to be that large.

## 5 · What these numbers do not tell you

* **Dynamic quantisation only touched `nn.Linear`.** Convolutions,
  normalisation and activations are untouched. On a CNN the saving is smaller
  than 4x.
* **This is not the device.** Your laptop's int8 path is not a
  microcontroller's. The size number transfers; the latency number does not.
* **Static quantisation needs calibration.** Quantising activations to a fixed
  range requires running representative data through the model first, and a
  badly chosen range costs real accuracy. Dynamic quantisation avoids that
  question by paying for it at runtime.
* **Accuracy on 150 samples has an uncertainty of a few percentage points.**
  Quote the test-set size next to any accuracy delta, or the delta means
  nothing.

## 5 · Float32 → int8, twice

Section 3 quantised a **classifier** and the accuracy barely moved. That is the
number everybody quotes, and it is true. It is also the wrong number for
L11's steering network, which is a **regression**.

The difference is not the code. It is the metric. An argmax absorbs a small
perturbation of every logit; a regression reports its output directly, so every
bit of rounding error lands in the answer. To see this you need the *same*
quantisation applied to *both* kinds of model, which `quantize_dynamic` cannot
give you cleanly (it also quantises the activations, and only at 8 bits). So
this section does the arithmetic by hand — it is the arithmetic of an
analogue-to-digital converter, and L6.2 slide 19 is exactly this.

`quantise_weight` maps a weight tensor onto `2**bits` uniformly spaced levels,
symmetric about zero (weights are centred, so the offset of the affine scheme
buys nothing here), and returns the *dequantised* tensor — the values the
integer hardware would actually multiply by. Per-tensor uses one scale for the
whole matrix; per-channel uses one scale per output row, which costs one extra
float per row and lets a small-magnitude row keep its resolution.

In [ ]:
import copy

def quantise_weight(w, bits=8, per_channel=False):
    """Round w onto 2**bits symmetric levels and return (dequantised w, scale)."""
    qmax = 2 ** (bits - 1) - 1
    amax = w.abs().amax(dim=1, keepdim=True) if per_channel else w.abs().max()
    scale = amax.clamp(min=1e-12) / qmax
    return torch.round(w / scale).clamp(-qmax - 1, qmax) * scale, scale

def quantise_model(model, bits=8, per_channel=False):
    """A deep copy of model with every nn.Linear weight quantised in place.
    Returns the copy and the number of scale factors it needed."""
    q = copy.deepcopy(model)
    n_scales = 0
    with torch.no_grad():
        for m in q.modules():
            if isinstance(m, nn.Linear):
                wq, scale = quantise_weight(m.weight, bits, per_channel)
                m.weight.copy_(wq)
                n_scales += scale.numel()
    return q, n_scales

# sanity: 8-bit per-tensor rounding error on a random matrix is ~ half a level
w = torch.randn(16, 16)
wq, sc = quantise_weight(w, bits=8)
print(f"level spacing {float(sc):.4f}   largest rounding error {float((w - wq).abs().max()):.4f}")

### The regression

The model is the one from notebook 02: the damped structural response, fitted
by Adam and then L-BFGS to a very low error. Train it, quantise its weights,
and compare — then quantise the classifier from section 1 with the *same*
function and the same bit width.

In [ ]:
# TODO 3 --- the same quantisation on a regressor and a classifier ------------------------------------
# Three `...` to replace:
#   line 1  ->  torch.optim.LBFGS(reg.parameters(), lr=1.0, max_iter=20)    the finishing optimiser
#   line 2  ->  reg_mse(quantise_model(reg, 8)[0])                          regression, int8 per tensor
#   line 3  ->  accuracy(quantise_model(model, 8)[0], X_rest, y_rest)       classifier, int8 per tensor
xr, yr = core.response_dataset(n=200)
Xr, Yr = core.to_tensor(xr), core.to_tensor(yr)

core.set_seed(0)
reg = core.MLP(hidden=(16, 16))
opt_r = torch.optim.Adam(reg.parameters(), lr=0.01)
for _ in range(2000):
    opt_r.zero_grad()
    nn.MSELoss()(reg(Xr), Yr).backward()
    opt_r.step()
opt_l = ...                                       # <- torch.optim.LBFGS(reg.parameters(), lr=1.0, max_iter=20)
def closure():
    opt_l.zero_grad()
    loss = nn.MSELoss()(reg(Xr), Yr)
    loss.backward()
    return loss
for _ in range(60):
    opt_l.step(closure)

def reg_mse(m):
    with torch.no_grad():
        return float(nn.MSELoss()(m(Xr), Yr))

q_tensor,  n_scales_tensor  = quantise_model(reg, 8)
q_channel, n_scales_channel = quantise_model(reg, 8, per_channel=True)

twice = {
    "regression fp32"         : reg_mse(reg),
    "regression int8"         : ...,              # <- reg_mse(quantise_model(reg, 8)[0])
    "regression int8/channel" : reg_mse(q_channel),
    "classifier fp32"         : accuracy(model, X_rest, y_rest),
    "classifier int8"         : ...,              # <- accuracy(quantise_model(model, 8)[0], X_rest, y_rest)
}
# ------------------------------------------------------------------------------

In [ ]:
ratio_tensor  = twice["regression int8"] / twice["regression fp32"]
ratio_channel = twice["regression int8/channel"] / twice["regression fp32"]
print(core.error_table(
    [["regression, MSE",       f"{twice['regression fp32']:.3e}",
                               f"{twice['regression int8']:.3e}",
                               f"{ratio_tensor:,.0f}x worse"],
     ["regression, per-channel", "-",
                               f"{twice['regression int8/channel']:.3e}",
                               f"{ratio_channel:,.0f}x worse  ({ratio_tensor/ratio_channel:.1f}x better "
                               f"than per-tensor, for {n_scales_channel - n_scales_tensor} extra floats)"],
     ["classifier, accuracy",  f"{twice['classifier fp32']:.3f}",
                               f"{twice['classifier int8']:.3f}",
                               f"{twice['classifier int8'] - twice['classifier fp32']:+.3f}"]],
    ["model, metric", "float32", "int8", "change"]))

### Signal-to-noise against bit width

Treat the float32 prediction as the signal and the quantised model's deviation
from it as the noise. An ideal uniform quantiser of a full-scale signal gives
SNR ≈ 6.02·b + 1.76 dB — the line every ADC datasheet quotes, and the line on
L6.2 slide 20. The measured points fall below it, because the *weights* are
quantised and the error is then pushed through two layers of tanh before it
reaches the output; the slope is what survives.

In [ ]:
BITS = list(range(2, 9))
with torch.no_grad():
    y32 = reg(Xr)
    signal = float(y32.var())
snr_tensor, snr_channel = [], []
for b in BITS:
    for per_channel, store in ((False, snr_tensor), (True, snr_channel)):
        q, _ = quantise_model(reg, b, per_channel)
        with torch.no_grad():
            noise = float(((q(Xr) - y32) ** 2).mean())
        store.append(10 * np.log10(signal / max(noise, 1e-30)))

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(BITS, [6.02 * b + 1.76 for b in BITS], ls="--", color="#888888", label="6.02 b + 1.76 (ideal)")
ax.plot(BITS, snr_tensor, "o-", color="#1f77b4", label="measured, per-tensor")
ax.plot(BITS, snr_channel, "s-", color="#2ca02c", label="measured, per-channel")
ax.set_xlabel("bits per weight"); ax.set_ylabel("SNR of the prediction, dB")
ax.legend(frameon=False); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()
for b, st, sc in zip(BITS, snr_tensor, snr_channel):
    print(f"{b} bits   per-tensor {st:5.1f} dB   per-channel {sc:5.1f} dB")

**What you should see.** The classifier loses almost nothing, exactly as in
section 3. The regression's error rises by orders of magnitude at the same bit
width, and per-channel scales claw a useful factor back for a handful of extra
floats. The SNR line climbs at roughly six decibels per bit, below the ideal.

That is the whole lesson of L6.2 slide 20 in one table: *when somebody tells
you a compression technique is nearly free, ask which metric they measured.*

## 7 · Save


In [ ]:
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "nb04_quantisation.npz")
np.savez(path,
         twice_names=np.array(list(twice)), twice_values=np.asarray(list(twice.values()), dtype=float),
         snr_bits=np.asarray(BITS), snr_tensor=np.asarray(snr_tensor), snr_channel=np.asarray(snr_channel),
         n_scales=np.asarray([n_scales_tensor, n_scales_channel]),
         base_bytes=base["bytes"], base_seconds=base["seconds"],
         base_accuracy=base["accuracy"], base_parameters=base["parameters"],
         quant_bytes=quant["bytes"], quant_seconds=quant["seconds"],
         quant_accuracy=quant["accuracy"],
         widths=np.asarray(small["width"]),
         small_bytes=np.asarray(small["bytes"], dtype=float),
         small_accuracy=np.asarray(small["accuracy"], dtype=float))
print("wrote", path)

## 8 · Before you move on

1. Your model is 4x smaller and no less accurate. What would you need to
   measure before telling a colleague it is 4x faster?
2. Under what circumstance is quantisation the *wrong* answer to "this model is
   too big"?
3. Dynamic quantisation needs no calibration data. What does it pay for that
   convenience, and when would you accept the extra work of static
   quantisation instead?
4. Ex_11 runs a network on a Jetson Nano at a frame rate you must measure.
   Which of this notebook's three numbers is the binding constraint there?

Next: **notebook 05**, the report.
4. Section 5 quantised *weights* only. Name one other tensor the deployed
   network will round, and say whether the regression or the classifier
   is more exposed to it.
